# Bộ thực thi đơn giản: tín hiệu 5m, khớp lệnh trên nến 1m

Quy tắc cố định, **không ứng biến**:
1. **Chỉ vào lệnh khi đủ confident.** Độ confident = *R kỳ vọng sau phí* của model cho hướng tốt hơn: `score = max(e_long, e_short) − phí/barrier`. Lệnh được vào khi score vượt ngưỡng (ngưỡng tuyệt đối, hoặc top % của 60 ngày trước — chỉ dùng quá khứ).
2. **Vào lệnh market** ở giá mở nến 1m đầu tiên sau khi nến 5m tín hiệu đóng.
3. **SL / TP đặt một lần** theo barrier của tín hiệu, không cập nhật giữa chừng.
4. **Giữ tối đa 4 giờ**, hết giờ thì đóng.
5. **Các lệnh độc lập**: mỗi tín hiệu đủ điều kiện là một lệnh riêng, có thể chồng nhau.

Ngưỡng và mẫu TP được **chọn trên 2022–2024** và **kiểm tra trên 2025–nay** (dữ liệu chưa dùng để chọn).

⚠ Bước bổ sung nến mới cần mạng (API công khai Binance). Đặt `TOP_UP = False` để chỉ dùng CSV.

In [ ]:
from pathlib import Path
import importlib
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "model_training":
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL_DIR = PROJECT_ROOT / "model_training"
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import market_data as md
import execution_m1 as ex
md, ex = importlib.reload(md), importlib.reload(ex)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## CONFIG

In [ ]:
# ============================== CONFIG ==============================
HIST = PROJECT_ROOT / "historical_data"
PRED_DIR = PROJECT_ROOT / "artifacts" / "5m_xgboost_signal_v2"
OUT = PROJECT_ROOT / "artifacts" / "execution_m1"
TOP_UP = True                                   # bổ sung nến + tín hiệu mới qua REST (cần mạng)
SPLIT = pd.Timestamp("2025-01-01", tz="UTC")    # chọn cấu hình trước mốc này, kiểm tra sau mốc này
RECENT_DAYS = 30

BASE = ex.ExecConfig(max_hold_min=240, fee_roundtrip=0.001, window_days=60, min_history_days=20)
# Lưới cấu hình: mẫu 1R (TP = SL = 1 barrier) và 2R (TP = 2, SL = 1), ngưỡng tương đối và tuyệt đối
GRID = []
for rr, tp in ((1, 1.0), (2, 2.0)):
    for top in (0.2, 0.1, 0.05, 0.02, 0.01):
        GRID.append(ex.ExecConfig(**{**BASE.__dict__, "rr": rr, "tp_mult": tp, "mode": "rolling", "top_frac": top}))
    for ms in (0.02, 0.05, 0.1):
        GRID.append(ex.ExecConfig(**{**BASE.__dict__, "rr": rr, "tp_mult": tp, "mode": "abs", "min_score": ms}))
# ====================================================================
OUT.mkdir(parents=True, exist_ok=True)
SPLIT_MS = int(SPLIT.value // 1_000_000)

## 1. Dữ liệu: nến 1m và tín hiệu

In [ ]:
t0 = time.time()
c1 = md.load_klines(HIST / "BTCUSDT_1m_all.csv", "1m", top_up=TOP_UP)
c1 = c1[c1["open_time_ms"] >= pd.Timestamp("2021-12-01", tz="UTC").value // 1_000_000].reset_index(drop=True)
gaps = int((np.diff(c1["open_time_ms"].to_numpy()) != ex.M1).sum())
M1 = {"t": c1["open_time_ms"].to_numpy(np.int64), "o": c1["open"].to_numpy(float), "h": c1["high"].to_numpy(float),
      "l": c1["low"].to_numpy(float), "c": c1["close"].to_numpy(float)}
print(f"1m: {len(c1):,} nến → {pd.to_datetime(M1['t'][-1], unit='ms', utc=True)} | gaps {gaps} | {time.time() - t0:.0f}s")

sig = ex.load_signals(PRED_DIR)
last_oos = int(sig["decision_ms"].max())
if TOP_UP:
    # tín hiệu cho các nến sau file OOS (bundle = fold cuối, train trước 2025 -> vẫn là out-of-sample)
    import modelv2 as mv2
    c5 = md.load_klines(HIST / "BTCUSDT_5m_all.csv", "5m")
    new_bars = int((c5["open_time_ms"] >= last_oos).sum())
    if new_bars > 0:
        fund_hist = pd.read_csv(HIST / "BTCUSDT_funding.csv")[["funding_time_ms", "funding_rate"]]
        funding = pd.concat([fund_hist, md.fetch_funding(int(fund_hist["funding_time_ms"].max()) + 1)])
        sub = c5[c5["open_time_ms"] >= last_oos - 45 * 86_400_000]
        frame = mv2.predict_frame(mv2.load_bundle(PRED_DIR / "models_final.joblib"), sub, last_n=new_bars + 1,
                                  funding=funding.drop_duplicates("funding_time_ms"))
        sig = ex.append_live(sig, frame)
print(f"Tín hiệu: {len(sig):,} nến 5m | OOS đến {pd.to_datetime(last_oos, unit='ms', utc=True)} | "
      f"tới {pd.to_datetime(sig['decision_ms'].max(), unit='ms', utc=True)}")

## 2. Độ confident có ý nghĩa không? (mọi nến, mẫu 1R)

Trước khi giao dịch: chia các nến theo score, so R kỳ vọng của model với R thực tế sau phí (từ nhãn 4h của chính nến đó). Nếu model đáng tin, cột `realized` phải tăng theo `predicted`, ở **mọi năm**.

In [ ]:
s1 = ex.score_signals(sig, ex.ExecConfig(rr=1))
lab = np.where(s1["side"] == 1, s1["label_r_long_1r"], s1["label_r_short_1r"]) - s1["fee_r"]
s1 = s1.assign(realized=lab, year=s1["time"].dt.year).dropna(subset=["realized"])
s1["bucket"] = pd.cut(s1["score"], [-1, -0.05, 0, 0.02, 0.05, 0.1, 1])
tab = s1.pivot_table(index="bucket", columns="year", values="realized", aggfunc="mean", observed=True)
tab["predicted"] = s1.groupby("bucket", observed=True)["score"].mean()
tab["bars"] = s1.groupby("bucket", observed=True).size()
display(tab.round(3))

## 3. Chọn cấu hình trên 2022–2024, kiểm tra trên 2025–nay

In [ ]:
t0 = time.time()
sw = ex.sweep(M1, sig, GRID, SPLIT_MS)
print(f"{len(GRID)} cấu hình | {time.time() - t0:.0f}s")
display(sw.round(3))
best_i = int(sw["sel_avg_R"].idxmax())
CFG = GRID[best_i]
print("Cấu hình tốt nhất trên giai đoạn chọn:", CFG.label)
print(f"  chọn  : {sw.loc[best_i, 'sel_trades']} lệnh, {sw.loc[best_i, 'sel_avg_R']:+.3f} R/lệnh")
print(f"  kiểm tra: {sw.loc[best_i, 'hold_trades']} lệnh, {sw.loc[best_i, 'hold_avg_R']:+.3f} R/lệnh "
      f"(KTC 90% {sw.loc[best_i, 'hold_ci_lo']:+.3f} … {sw.loc[best_i, 'hold_ci_hi']:+.3f})")

## 4. Cấu hình đã chọn: chi tiết

In [ ]:
scored = ex.score_signals(sig, CFG)
trades = ex.simulate(M1, scored, CFG)
parts = {"chọn (trước 2025)": trades[trades["entry_ms"] < SPLIT_MS], "kiểm tra (2025–nay)": trades[trades["entry_ms"] >= SPLIT_MS],
         "toàn kỳ": trades}
display(pd.DataFrame({k: ex.performance(v) for k, v in parts.items()}).T.round(3))
print("Theo năm:")
display(ex.by_period(trades, "Y")[["trades", "win_rate", "avg_R", "ci90_lo", "ci90_hi", "total_R", "max_dd_R", "tp", "sl", "timeout", "max_open"]].round(3))
print("Model có quá tự tin không (R kỳ vọng vs thực tế, theo nhóm score):")
display(ex.calibration(scored, trades).round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [2, 1]})
for k, tr in parts.items():
    if k != "toàn kỳ" and len(tr):
        axes[0].plot(tr["exit_time"], tr["R"].cumsum(), lw=1.2, label=f"{k}: {len(tr)} lệnh, {tr['R'].sum():+.0f}R")
axes[0].axhline(0, color="k", lw=0.7)
axes[0].set(title=f"R cộng dồn sau phí — {CFG.label}", ylabel="cum R")
axes[0].legend()
q = ex.by_period(trades, "Q")
axes[1].bar(q.index, q["total_R"], color=np.where(q["total_R"] > 0, "tab:green", "tab:red"))
axes[1].set(title="Tổng R theo quý", ylabel="R")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 5. 30 ngày gần nhất

In [ ]:
recent_ms = int(M1["t"][-1]) - RECENT_DAYS * 86_400_000
rt = trades[trades["entry_ms"] >= recent_ms]
display(pd.DataFrame({"30 ngày": ex.performance(rt)}).T.round(3))
cols = ["signal_time", "side", "entry", "sl", "tp", "exit", "reason", "minutes", "score", "threshold", "R"]
display(rt[cols].tail(40).round(4))
px = c1[c1["open_time_ms"] >= recent_ms]
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(pd.to_datetime(px["open_time_ms"], unit="ms", utc=True), px["close"], lw=0.5, color="gray")
for r in rt.itertuples():
    ax.plot([r.entry_time, r.exit_time], [r.entry, r.exit], color="tab:green" if r.R > 0 else "tab:red", lw=2)
    ax.scatter([r.entry_time], [r.entry], marker="^" if r.side == 1 else "v", color="k", s=14, zorder=3)
ax.set(title="30 ngày gần nhất: lệnh của cấu hình đã chọn (xanh = lãi, đỏ = lỗ)")
plt.show()

## 6. Lưu kết quả

In [ ]:
sw.to_csv(OUT / "sweep.csv", index=False)
trades.to_csv(OUT / "trades.csv", index=False)
ex.by_period(trades, "Q").to_csv(OUT / "quarterly.csv")
pd.Series(CFG.__dict__).to_json(OUT / "chosen_config.json", indent=2)
print("Đã lưu vào", OUT)